# 06 revenue feature engineering

This GitHub copy has been cleaned for readability: duplicate/empty cells were removed and saved cell outputs were cleared. Code and analysis workflow are retained.

**Execution note:** Original project paths may reference Google Drive/Colab. Update paths and provide required source files/API credentials in your own environment before rerunning.

In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np

# Mount Google Drive
drive.mount("/content/drive")

# Project root
PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

# Input
REVENUE_FILE = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "ICMB_130_REVENUE_DATA.csv"
)

# Output
FEATURED_FILE = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "ICMB_130_REVENUE_FEATURES.csv"
)

print("=" * 60)
print("NOTEBOOK 06 — GOOGLE DRIVE / PROJECT CHECK")
print("=" * 60)

print("Project root exists:", os.path.exists(PROJECT_ROOT))
print("Revenue dataset exists:", os.path.exists(REVENUE_FILE))
print("Output directory exists:", os.path.exists(os.path.dirname(FEATURED_FILE)))

In [ ]:
print("=" * 60)
print("NOTEBOOK 06 — LOADING REVENUE DATASET")
print("=" * 60)

revenue_df = pd.read_csv(REVENUE_FILE)

print("Dataset loaded successfully.")
print("Rows:", len(revenue_df))
print("Columns:", len(revenue_df.columns))

print("\nColumns:")
for i, col in enumerate(revenue_df.columns, 1):
    print(f"{i}. {col}")

print("\nData types:")
print(revenue_df.dtypes)

print("\nFirst 5 records:")
display(revenue_df.head())

In [ ]:
print("=" * 60)
print("NOTEBOOK 06 — PRE-FEATURE ENGINEERING VALIDATION")
print("=" * 60)

# Expected columns
expected_columns = [
    "movie_record_id",
    "year",
    "rank",
    "movie_title",
    "language",
    "india_gross_inr_cr",
    "tmdb_id",
    "tmdb_title",
    "tmdb_budget_usd",
    "tmdb_worldwide_revenue_usd"
]

# ------------------------------------------------------------
# 1. Schema validation
# ------------------------------------------------------------

missing_columns = set(expected_columns) - set(revenue_df.columns)
extra_columns = set(revenue_df.columns) - set(expected_columns)

print("Schema:")
print("Expected columns:", len(expected_columns))
print("Actual columns:", len(revenue_df.columns))
print("Missing columns:", missing_columns)
print("Extra columns:", extra_columns)

# ------------------------------------------------------------
# 2. Record validation
# ------------------------------------------------------------

print("\nRecord integrity:")
print("Rows:", len(revenue_df))
print("Unique movie_record_id:", revenue_df["movie_record_id"].nunique())
print("Unique tmdb_id:", revenue_df["tmdb_id"].nunique())

print("Duplicate movie_record_id:",
      revenue_df["movie_record_id"].duplicated().sum())

print("Duplicate tmdb_id:",
      revenue_df["tmdb_id"].duplicated().sum())

# ------------------------------------------------------------
# 3. Missing-value validation
# ------------------------------------------------------------

print("\nMissing values:")
for col in expected_columns:
    print(f"{col}: {revenue_df[col].isna().sum()}")

# ------------------------------------------------------------
# 4. Year validation
# ------------------------------------------------------------

print("\nYear range:")
print("Minimum year:", revenue_df["year"].min())
print("Maximum year:", revenue_df["year"].max())

# ------------------------------------------------------------
# 5. Revenue validation
# ------------------------------------------------------------

print("\nRevenue values:")
print("India Gross minimum:", revenue_df["india_gross_inr_cr"].min())
print("India Gross maximum:", revenue_df["india_gross_inr_cr"].max())

print("TMDB Worldwide Revenue minimum:",
      revenue_df["tmdb_worldwide_revenue_usd"].min())

print("TMDB Worldwide Revenue maximum:",
      revenue_df["tmdb_worldwide_revenue_usd"].max())

print("TMDB Budget minimum:",
      revenue_df["tmdb_budget_usd"].min())

print("TMDB Budget maximum:",
      revenue_df["tmdb_budget_usd"].max())

# ------------------------------------------------------------
# Final check
# ------------------------------------------------------------

schema_pass = (
    len(revenue_df.columns) == len(expected_columns)
    and not missing_columns
    and not extra_columns
)

integrity_pass = (
    len(revenue_df) == 130
    and revenue_df["movie_record_id"].nunique() == 130
    and revenue_df["tmdb_id"].nunique() == 130
    and revenue_df["movie_record_id"].duplicated().sum() == 0
    and revenue_df["tmdb_id"].duplicated().sum() == 0
)

missing_pass = all(
    revenue_df[col].isna().sum() == 0
    for col in expected_columns
)

year_pass = (
    revenue_df["year"].min() == 2000
    and revenue_df["year"].max() == 2025
)

financial_pass = (
    (revenue_df["india_gross_inr_cr"] > 0).all()
    and (revenue_df["tmdb_budget_usd"] >= 0).all()
    and (revenue_df["tmdb_worldwide_revenue_usd"] >= 0).all()
)

print("\n" + "=" * 60)

if schema_pass:
    print("✅ PASS — Schema validation")
else:
    print("❌ FAIL — Schema validation")

if integrity_pass:
    print("✅ PASS — Record integrity")
else:
    print("❌ FAIL — Record integrity")

if missing_pass:
    print("✅ PASS — Missing-value validation")
else:
    print("❌ FAIL — Missing-value validation")

if year_pass:
    print("✅ PASS — Year validation")
else:
    print("❌ FAIL — Year validation")

if financial_pass:
    print("✅ PASS — Financial value validation")
else:
    print("❌ FAIL — Financial value validation")

all_pass = (
    schema_pass
    and integrity_pass
    and missing_pass
    and year_pass
    and financial_pass
)

print("=" * 60)

if all_pass:
    print("✅ NOTEBOOK 06 — PRE-FEATURE VALIDATION PASSED")
    print("Dataset is ready for feature engineering.")
else:
    print("❌ NOTEBOOK 06 — PRE-FEATURE VALIDATION FAILED")
    print("STOP — Do not continue until the issue is resolved.")

In [ ]:
print("=" * 60)
print("NOTEBOOK 06 — REVENUE FEATURE ENGINEERING")
print("=" * 60)

# Work on a copy so the validated input dataframe is not modified
features_df = revenue_df.copy()

# ============================================================
# 1. Convert TMDB USD values to INR crore
# ============================================================
# Fixed project conversion assumption:
# 1 USD = ₹83
# Therefore:
# USD → INR crore = USD × 83 / 10,000,000

USD_TO_INR = 83

features_df["tmdb_budget_inr_cr"] = (
    features_df["tmdb_budget_usd"] * USD_TO_INR / 10_000_000
)

features_df["tmdb_worldwide_revenue_inr_cr"] = (
    features_df["tmdb_worldwide_revenue_usd"] * USD_TO_INR / 10_000_000
)

# ============================================================
# 2. Validity flags for TMDB financial data
# ============================================================

features_df["budget_available"] = (
    features_df["tmdb_budget_usd"] > 0
)

features_df["worldwide_revenue_available"] = (
    features_df["tmdb_worldwide_revenue_usd"] > 0
)

# ============================================================
# 3. Revenue-to-budget ratio
# ============================================================

features_df["revenue_budget_ratio"] = np.where(
    features_df["budget_available"]
    & features_df["worldwide_revenue_available"],
    features_df["tmdb_worldwide_revenue_usd"]
    / features_df["tmdb_budget_usd"],
    np.nan
)

# ============================================================
# 4. India Gross as percentage of TMDB worldwide revenue
# ============================================================

features_df["india_gross_share_pct"] = np.where(
    features_df["worldwide_revenue_available"],
    (
        features_df["india_gross_inr_cr"]
        / features_df["tmdb_worldwide_revenue_inr_cr"]
    ) * 100,
    np.nan
)

# ============================================================
# 5. Log-transformed revenue features
# ============================================================
# Useful later for statistical analysis/modeling because
# movie revenues are highly right-skewed.

features_df["log_india_gross"] = np.log1p(
    features_df["india_gross_inr_cr"]
)

features_df["log_worldwide_revenue"] = np.where(
    features_df["worldwide_revenue_available"],
    np.log1p(features_df["tmdb_worldwide_revenue_inr_cr"]),
    np.nan
)

features_df["log_budget"] = np.where(
    features_df["budget_available"],
    np.log1p(features_df["tmdb_budget_inr_cr"]),
    np.nan
)

print("Feature engineering completed.")

print("\nOriginal columns:", len(revenue_df.columns))
print("New columns:", len(features_df.columns))
print("Features added:", len(features_df.columns) - len(revenue_df.columns))

print("\nNew feature columns:")
new_columns = [
    col for col in features_df.columns
    if col not in revenue_df.columns
]

for col in new_columns:
    print("-", col)

print("\nFeature preview:")
display(
    features_df[
        [
            "movie_record_id",
            "movie_title",
            "india_gross_inr_cr",
            "tmdb_budget_usd",
            "tmdb_worldwide_revenue_usd",
            "tmdb_budget_inr_cr",
            "tmdb_worldwide_revenue_inr_cr",
            "budget_available",
            "worldwide_revenue_available",
            "revenue_budget_ratio",
            "india_gross_share_pct",
            "log_india_gross",
            "log_worldwide_revenue",
            "log_budget"
        ]
    ].head(10)
)

In [ ]:
print("=" * 60)
print("NOTEBOOK 06 — DERIVED FEATURE VALIDATION")
print("=" * 60)

# ============================================================
# 1. Row integrity
# ============================================================

print("Rows:")
print("Original:", len(revenue_df))
print("Featured:", len(features_df))

# ============================================================
# 2. Check original columns were not changed
# ============================================================

original_columns_unchanged = True

for col in revenue_df.columns:
    if not revenue_df[col].equals(features_df[col]):
        original_columns_unchanged = False
        print("Changed original column:", col)

print("\nOriginal columns unchanged:", original_columns_unchanged)

# ============================================================
# 3. Check derived columns
# ============================================================

expected_new_features = [
    "tmdb_budget_inr_cr",
    "tmdb_worldwide_revenue_inr_cr",
    "budget_available",
    "worldwide_revenue_available",
    "revenue_budget_ratio",
    "india_gross_share_pct",
    "log_india_gross",
    "log_worldwide_revenue",
    "log_budget"
]

missing_features = [
    col for col in expected_new_features
    if col not in features_df.columns
]

print("\nDerived features expected:", len(expected_new_features))
print("Derived features found:",
      len(expected_new_features) - len(missing_features))
print("Missing derived features:", missing_features)

# ============================================================
# 4. Availability flag validation
# ============================================================

budget_flag_correct = (
    features_df["budget_available"]
    == (features_df["tmdb_budget_usd"] > 0)
).all()

worldwide_flag_correct = (
    features_df["worldwide_revenue_available"]
    == (features_df["tmdb_worldwide_revenue_usd"] > 0)
).all()

print("\nAvailability flags:")
print("Budget flag correct:", budget_flag_correct)
print("Worldwide revenue flag correct:", worldwide_flag_correct)

# ============================================================
# 5. Converted INR values validation
# ============================================================

expected_budget_inr = (
    features_df["tmdb_budget_usd"] * USD_TO_INR / 10_000_000
)

expected_worldwide_inr = (
    features_df["tmdb_worldwide_revenue_usd"]
    * USD_TO_INR / 10_000_000
)

budget_conversion_correct = np.allclose(
    features_df["tmdb_budget_inr_cr"],
    expected_budget_inr
)

worldwide_conversion_correct = np.allclose(
    features_df["tmdb_worldwide_revenue_inr_cr"],
    expected_worldwide_inr
)

print("\nCurrency conversion:")
print("Budget conversion correct:", budget_conversion_correct)
print("Worldwide revenue conversion correct:",
      worldwide_conversion_correct)

# ============================================================
# 6. Ratio validation
# ============================================================

valid_ratio_rows = (
    features_df["budget_available"]
    & features_df["worldwide_revenue_available"]
)

ratio_valid = (
    features_df.loc[valid_ratio_rows, "revenue_budget_ratio"]
    > 0
).all()

invalid_ratio_rows = features_df.loc[
    ~valid_ratio_rows,
    "revenue_budget_ratio"
].notna().sum()

print("\nRevenue-to-budget ratio:")
print("Valid ratio rows:", valid_ratio_rows.sum())
print("Positive ratios:", ratio_valid)
print("Invalid ratios where financial data unavailable:",
      invalid_ratio_rows)

# ============================================================
# 7. Log feature validation
# ============================================================

log_india_valid = (
    np.isfinite(features_df["log_india_gross"])
).all()

log_worldwide_valid = np.isfinite(
    features_df.loc[
        features_df["worldwide_revenue_available"],
        "log_worldwide_revenue"
    ]
).all()

log_budget_valid = np.isfinite(
    features_df.loc[
        features_df["budget_available"],
        "log_budget"
    ]
).all()

print("\nLog features:")
print("India Gross log valid:", log_india_valid)
print("Worldwide Revenue log valid:", log_worldwide_valid)
print("Budget log valid:", log_budget_valid)

# ============================================================
# 8. Negative-value validation
# ============================================================

negative_budget = (
    features_df["tmdb_budget_inr_cr"] < 0
).sum()

negative_worldwide = (
    features_df["tmdb_worldwide_revenue_inr_cr"] < 0
).sum()

negative_ratio = (
    features_df["revenue_budget_ratio"].dropna() < 0
).sum()

print("\nNegative derived values:")
print("Negative budget:", negative_budget)
print("Negative worldwide revenue:", negative_worldwide)
print("Negative revenue-budget ratios:", negative_ratio)

# ============================================================
# FINAL RESULT
# ============================================================

all_feature_checks = (
    len(features_df) == len(revenue_df)
    and original_columns_unchanged
    and len(missing_features) == 0
    and budget_flag_correct
    and worldwide_flag_correct
    and budget_conversion_correct
    and worldwide_conversion_correct
    and ratio_valid
    and invalid_ratio_rows == 0
    and log_india_valid
    and log_worldwide_valid
    and log_budget_valid
    and negative_budget == 0
    and negative_worldwide == 0
    and negative_ratio == 0
)

print("\n" + "=" * 60)

if all_feature_checks:
    print("✅ NOTEBOOK 06 — DERIVED FEATURE VALIDATION PASSED")
    print("Feature dataset is ready for final schema validation.")
else:
    print("❌ NOTEBOOK 06 — DERIVED FEATURE VALIDATION FAILED")
    print("STOP — Do not save the feature dataset yet.")

print("=" * 60)

In [ ]:
# ============================================================
# NOTEBOOK 06 — FINAL FEATURE ENGINEERING + SCHEMA VALIDATION
# ============================================================

import os
import numpy as np
import pandas as pd

# ============================================================
# 1. PATHS
# ============================================================

INPUT_PATH = (
    "/content/drive/MyDrive/"
    "IndianCinemaTrendAnalytics/01_data/processed/"
    "ICMB_130_REVENUE_DATA.csv"
)

OUTPUT_PATH = (
    "/content/drive/MyDrive/"
    "IndianCinemaTrendAnalytics/01_data/processed/"
    "ICMB_130_FEATURES.csv"
)

# ============================================================
# 2. LOAD REVENUE DATASET
# ============================================================

df = pd.read_csv(INPUT_PATH)

print("=" * 60)
print("NOTEBOOK 06 — FEATURE DATASET PREPARATION")
print("=" * 60)

print("Input rows:", len(df))
print("Input columns:", len(df.columns))

assert len(df) == 130

# ============================================================
# 3. CREATE FEATURE DATAFRAME
# ============================================================

feature_df = df.copy()

# ------------------------------------------------------------
# USD → INR conversion
# Project conversion rate: 1 USD = ₹83
# ------------------------------------------------------------

USD_TO_INR = 83

feature_df["tmdb_budget_inr_cr"] = (
    feature_df["tmdb_budget_usd"] * USD_TO_INR / 10_000_000
)

feature_df["tmdb_worldwide_revenue_inr_cr"] = (
    feature_df["tmdb_worldwide_revenue_usd"] * USD_TO_INR / 10_000_000
)

# ------------------------------------------------------------
# Availability flags
# ------------------------------------------------------------

feature_df["budget_available"] = (
    feature_df["tmdb_budget_usd"] > 0
)

feature_df["worldwide_revenue_available"] = (
    feature_df["tmdb_worldwide_revenue_usd"] > 0
)

# ------------------------------------------------------------
# Revenue / Budget ratio
# Only calculate where both values are available
# ------------------------------------------------------------

feature_df["revenue_budget_ratio"] = np.where(
    (
        (feature_df["tmdb_budget_usd"] > 0) &
        (feature_df["tmdb_worldwide_revenue_usd"] > 0)
    ),
    feature_df["tmdb_worldwide_revenue_usd"] /
    feature_df["tmdb_budget_usd"],
    np.nan
)

# ------------------------------------------------------------
# India gross share of worldwide revenue
# Only calculate where worldwide revenue is available
# ------------------------------------------------------------

feature_df["india_gross_share_pct"] = np.where(
    feature_df["tmdb_worldwide_revenue_inr_cr"] > 0,
    (
        feature_df["india_gross_inr_cr"] /
        feature_df["tmdb_worldwide_revenue_inr_cr"]
    ) * 100,
    np.nan
)

# ------------------------------------------------------------
# Log transformations
# ------------------------------------------------------------

feature_df["log_india_gross"] = np.log1p(
    feature_df["india_gross_inr_cr"]
)

feature_df["log_worldwide_revenue"] = np.where(
    feature_df["tmdb_worldwide_revenue_usd"] > 0,
    np.log1p(feature_df["tmdb_worldwide_revenue_usd"]),
    np.nan
)

feature_df["log_budget"] = np.where(
    feature_df["tmdb_budget_usd"] > 0,
    np.log1p(feature_df["tmdb_budget_usd"]),
    np.nan
)

# ============================================================
# 4. FINAL EXPECTED SCHEMA
# ============================================================

EXPECTED_COLUMNS = [
    "movie_record_id",
    "year",
    "rank",
    "movie_title",
    "language",
    "india_gross_inr_cr",
    "tmdb_id",
    "tmdb_title",
    "tmdb_budget_usd",
    "tmdb_worldwide_revenue_usd",
    "tmdb_budget_inr_cr",
    "tmdb_worldwide_revenue_inr_cr",
    "budget_available",
    "worldwide_revenue_available",
    "revenue_budget_ratio",
    "india_gross_share_pct",
    "log_india_gross",
    "log_worldwide_revenue",
    "log_budget"
]

# ============================================================
# 5. FINAL SCHEMA VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("NOTEBOOK 06 — FINAL FEATURE DATASET SCHEMA VALIDATION")
print("=" * 60)

print("Expected rows:", 130)
print("Actual rows:", len(feature_df))

assert len(feature_df) == 130

print("\nExpected columns:", len(EXPECTED_COLUMNS))
print("Actual columns:", len(feature_df.columns))

missing_columns = set(EXPECTED_COLUMNS) - set(feature_df.columns)
extra_columns = set(feature_df.columns) - set(EXPECTED_COLUMNS)

print("Missing columns:", missing_columns)
print("Extra columns:", extra_columns)

assert not missing_columns
assert not extra_columns

column_order_correct = (
    list(feature_df.columns) == EXPECTED_COLUMNS
)

print("Column order correct:", column_order_correct)

assert column_order_correct

# ============================================================
# 6. RECORD INTEGRITY
# ============================================================

print("\nRecord integrity:")

print(
    "Unique movie_record_id:",
    feature_df["movie_record_id"].nunique()
)

print(
    "Unique tmdb_id:",
    feature_df["tmdb_id"].nunique()
)

print(
    "Duplicate movie_record_id:",
    feature_df["movie_record_id"].duplicated().sum()
)

print(
    "Duplicate tmdb_id:",
    feature_df["tmdb_id"].duplicated().sum()
)

assert feature_df["movie_record_id"].nunique() == 130
assert feature_df["tmdb_id"].nunique() == 130
assert feature_df["movie_record_id"].duplicated().sum() == 0
assert feature_df["tmdb_id"].duplicated().sum() == 0

# ============================================================
# 7. REQUIRED FIELD VALIDATION
# ============================================================

required_columns = [
    "movie_record_id",
    "year",
    "rank",
    "movie_title",
    "language",
    "india_gross_inr_cr",
    "tmdb_id",
    "tmdb_title",
    "tmdb_budget_usd",
    "tmdb_worldwide_revenue_usd"
]

missing_required = feature_df[required_columns].isna().sum()

print("\nRequired-field missing values:")
print(missing_required)

assert missing_required.sum() == 0

# ============================================================
# 8. FEATURE VALIDATION
# ============================================================

print("\nDerived feature validation:")

print(
    "Budget availability flags:",
    feature_df["budget_available"].dtype
)

print(
    "Worldwide revenue availability flags:",
    feature_df["worldwide_revenue_available"].dtype
)

assert (
    feature_df["budget_available"]
    == (feature_df["tmdb_budget_usd"] > 0)
).all()

assert (
    feature_df["worldwide_revenue_available"]
    == (feature_df["tmdb_worldwide_revenue_usd"] > 0)
).all()

# Currency conversion
budget_conversion_check = np.isclose(
    feature_df["tmdb_budget_inr_cr"],
    feature_df["tmdb_budget_usd"] * 83 / 10_000_000
).all()

worldwide_conversion_check = np.isclose(
    feature_df["tmdb_worldwide_revenue_inr_cr"],
    feature_df["tmdb_worldwide_revenue_usd"] * 83 / 10_000_000
).all()

print("Budget conversion correct:", budget_conversion_check)
print(
    "Worldwide revenue conversion correct:",
    worldwide_conversion_check
)

assert budget_conversion_check
assert worldwide_conversion_check

# ============================================================
# 9. FINAL VALIDATION SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("NOTEBOOK 06 — FINAL VALIDATION SUMMARY")
print("=" * 60)

print("✅ PASS — 130 records")
print("✅ PASS — 19 columns")
print("✅ PASS — Exact column order")
print("✅ PASS — Required fields complete")
print("✅ PASS — Record integrity")
print("✅ PASS — No duplicate movie records")
print("✅ PASS — No duplicate TMDB IDs")
print("✅ PASS — Availability flags")
print("✅ PASS — Currency conversion")
print("✅ PASS — Derived features")

print("=" * 60)
print("✅ NOTEBOOK 06 — FINAL FEATURE DATASET VALIDATION PASSED")
print("=" * 60)

# ============================================================
# 10. SAVE FINAL FEATURE DATASET
# ============================================================

feature_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nFinal feature dataset saved:")
print(OUTPUT_PATH)

print("File exists:", os.path.exists(OUTPUT_PATH))
print("Rows saved:", len(feature_df))
print("Columns saved:", len(feature_df.columns))